In [1]:
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, util

# Device
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

# Paths
WIKIR_DIR = Path("/Users/uni/homework13April/wikIR1k")
MIRAGE_DIR = Path("/Users/uni/homework13April/mirage")
EMB_DIR = Path("/Users/uni/homework13April/embeddings")
EMB_DIR.mkdir(exist_ok=True)

print("All imports OK")

Using device: mps
All imports OK


In [2]:
# Load WikIR data
docs_df = pd.read_csv(WIKIR_DIR / "documents.csv")
doc_ids = docs_df["id_right"].tolist()
doc_texts = docs_df["text_right"].tolist()

test_queries_df = pd.read_csv(WIKIR_DIR / "test" / "queries.csv")
test_qids = test_queries_df["id_left"].tolist()
test_qtexts = test_queries_df["text_left"].tolist()

# Load qrels
def load_qrels(path):
    qrels = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split("\t")
            qid, _, docid, rel = int(parts[0]), parts[1], int(parts[2]), int(parts[3])
            if qid not in qrels:
                qrels[qid] = {}
            qrels[qid][docid] = rel
    return qrels

test_qrels = load_qrels(WIKIR_DIR / "test" / "qrels")

print(f"Corpus: {len(doc_ids):,} docs")
print(f"Test queries: {len(test_qids)}")
print(f"Test qrels queries: {len(test_qrels)}")

Corpus: 369,721 docs
Test queries: 100
Test qrels queries: 100


In [3]:
# Load MIRAGE data
with open(MIRAGE_DIR / "dataset.json") as f:
    mirage_dataset = json.load(f)

with open(MIRAGE_DIR / "doc_pool.json") as f:
    mirage_doc_pool = json.load(f)

with open(MIRAGE_DIR / "oracle.json") as f:
    mirage_oracle = json.load(f)

# Group docs by query_id
from collections import defaultdict
mirage_docs_by_query = defaultdict(list)
for doc in mirage_doc_pool:
    mirage_docs_by_query[doc["mapped_id"]].append(doc)

print(f"MIRAGE queries: {len(mirage_dataset)}")
print(f"MIRAGE doc pool: {len(mirage_doc_pool)} ({len(mirage_doc_pool)//len(mirage_dataset)} docs/query)")
print(f"MIRAGE oracle entries: {len(mirage_oracle)}")

MIRAGE queries: 7560
MIRAGE doc pool: 37800 (5 docs/query)
MIRAGE oracle entries: 7560


In [4]:
# Evaluation functions: P@k, MAP@20, nDCG@20

def precision_at_k(ranked_doc_ids, relevant_docs, k):
    top_k = ranked_doc_ids[:k]
    hits = sum(1 for d in top_k if d in relevant_docs and relevant_docs[d] > 0)
    return hits / k

def ap_at_k(ranked_doc_ids, relevant_docs, k):
    hits, score = 0, 0.0
    for i, doc in enumerate(ranked_doc_ids[:k]):
        if doc in relevant_docs and relevant_docs[doc] > 0:
            hits += 1
            score += hits / (i + 1)
    num_rel = sum(1 for r in relevant_docs.values() if r > 0)
    return score / min(num_rel, k) if num_rel > 0 else 0.0

def dcg_at_k(ranked_doc_ids, relevant_docs, k):
    score = 0.0
    for i, doc in enumerate(ranked_doc_ids[:k]):
        rel = relevant_docs.get(doc, 0)
        score += rel / np.log2(i + 2)
    return score

def ndcg_at_k(ranked_doc_ids, relevant_docs, k):
    actual = dcg_at_k(ranked_doc_ids, relevant_docs, k)
    ideal_rels = sorted(relevant_docs.values(), reverse=True)[:k]
    ideal = sum(r / np.log2(i + 2) for i, r in enumerate(ideal_rels))
    return actual / ideal if ideal > 0 else 0.0

def evaluate(results_per_query, qrels):
    """
    results_per_query: dict {qid: [doc_id, doc_id, ...]} ranked list
    qrels: dict {qid: {doc_id: relevance}}
    """
    p1, p10, p20, map20, ndcg20 = [], [], [], [], []
    for qid, ranked in results_per_query.items():
        rel = qrels.get(qid, {})
        p1.append(precision_at_k(ranked, rel, 1))
        p10.append(precision_at_k(ranked, rel, 10))
        p20.append(precision_at_k(ranked, rel, 20))
        map20.append(ap_at_k(ranked, rel, 20))
        ndcg20.append(ndcg_at_k(ranked, rel, 20))
    return {
        "P@1":    round(np.mean(p1), 4),
        "P@10":   round(np.mean(p10), 4),
        "P@20":   round(np.mean(p20), 4),
        "MAP@20": round(np.mean(map20), 4),
        "nDCG@20":round(np.mean(ndcg20), 4),
    }

print("Evaluation functions ready")

Evaluation functions ready


In [5]:
def encode_corpus_cached(model, model_name, doc_texts, device):
    """Encode corpus and cache to disk. Loads from cache if exists."""
    safe_name = model_name.replace("/", "_")
    cache_path = EMB_DIR / f"wikir_docs_{safe_name}.npy"
    if cache_path.exists():
        print(f"Loading cached embeddings from {cache_path}")
        return np.load(str(cache_path))
    print(f"Encoding {len(doc_texts):,} docs with {model_name} ...")
    embeddings = model.encode(
        doc_texts,
        batch_size=64,
        show_progress_bar=True,
        device=device,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    np.save(str(cache_path), embeddings)
    print(f"Saved to {cache_path}")
    return embeddings

def wikir_retrieval(model, model_name, doc_ids, doc_texts, query_ids, query_texts, device, top_k=100):
    """Run semantic search on WikIR and return ranked results."""
    doc_embs = encode_corpus_cached(model, model_name, doc_texts, device)

    print(f"Encoding {len(query_texts)} queries...")
    query_embs = model.encode(
        query_texts,
        show_progress_bar=True,
        device=device,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

    print("Running semantic search...")
    hits = util.semantic_search(query_embs, doc_embs, top_k=top_k)

    results = {}
    for i, qid in enumerate(query_ids):
        results[qid] = [doc_ids[h["corpus_id"]] for h in hits[i]]
    return results

print("Helper functions ready")

Helper functions ready


In [6]:
## Model 1: multi-qa-MiniLM-L6-cos-v1 — WikIR
MODEL1_NAME = "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"
print(f"Loading {MODEL1_NAME}...")
model1 = SentenceTransformer(MODEL1_NAME, device=device)
print("Model loaded.")

wikir_results_m1 = wikir_retrieval(model1, MODEL1_NAME, doc_ids, doc_texts, test_qids, test_qtexts, device)
metrics_wikir_m1 = evaluate(wikir_results_m1, test_qrels)
print("\nWikIR — multi-qa-MiniLM-L6-cos-v1:")
print(metrics_wikir_m1)

Loading sentence-transformers/multi-qa-MiniLM-L6-cos-v1...
Model loaded.
Loading cached embeddings from /Users/uni/homework13April/embeddings/wikir_docs_sentence-transformers_multi-qa-MiniLM-L6-cos-v1.npy
Encoding 100 queries...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Running semantic search...

WikIR — multi-qa-MiniLM-L6-cos-v1:
{'P@1': 0.63, 'P@10': 0.184, 'P@20': 0.1215, 'MAP@20': 0.1368, 'nDCG@20': 0.3547}


In [7]:
def mirage_retrieval(model, model_name, mirage_dataset, mirage_docs_by_query, mirage_oracle, device):
    """Rank candidate docs per query using semantic search. Deduplicates by doc_name."""
    results = {}
    qrels = {}

    for entry in tqdm(mirage_dataset, desc=f"MIRAGE {model_name.split('/')[-1]}"):
        qid = entry["query_id"]
        query = entry["query"]
        candidate_docs = mirage_docs_by_query[qid]

        if not candidate_docs:
            continue

        doc_texts_q = [d["doc_chunk"] for d in candidate_docs]
        doc_names = [d["doc_name"] for d in candidate_docs]

        query_emb = model.encode(query, normalize_embeddings=True, convert_to_numpy=True)
        doc_embs = model.encode(doc_texts_q, normalize_embeddings=True, convert_to_numpy=True)

        hits = util.semantic_search(query_emb, doc_embs, top_k=len(candidate_docs))

        # Deduplicate by doc_name, keeping highest-scoring chunk per document
        seen = set()
        ranked_names = []
        for h in hits[0]:
            name = doc_names[h["corpus_id"]]
            if name not in seen:
                seen.add(name)
                ranked_names.append(name)

        results[qid] = ranked_names

        # Build qrels using unique doc_names
        relevant_name = mirage_oracle[qid]["doc_name"] if qid in mirage_oracle else None
        unique_names = list(dict.fromkeys(doc_names))  # preserve order, deduplicate
        qrels[qid] = {name: (1 if name == relevant_name else 0) for name in unique_names}

    return results, qrels

print("MIRAGE retrieval function ready")

MIRAGE retrieval function ready


In [8]:
## Model 1: multi-qa-MiniLM-L6-cos-v1 — MIRAGE
mirage_results_m1, mirage_qrels = mirage_retrieval(model1, MODEL1_NAME, mirage_dataset, mirage_docs_by_query, mirage_oracle, device)
metrics_mirage_m1 = evaluate(mirage_results_m1, mirage_qrels)
print("\nMIRAGE — multi-qa-MiniLM-L6-cos-v1:")
print(metrics_mirage_m1)

MIRAGE multi-qa-MiniLM-L6-cos-v1: 100%|██████████| 7560/7560 [05:21<00:00, 23.54it/s]


MIRAGE — multi-qa-MiniLM-L6-cos-v1:
{'P@1': 0.8774, 'P@10': 0.1, 'P@20': 0.05, 'MAP@20': 0.9331, 'nDCG@20': 0.9503}


In [9]:
import gc
del model1
gc.collect()
print("model1 freed")

model1 freed


In [10]:
## Model 2: msmarco-distilbert-dot-v5 — WikIR
MODEL2_NAME = "sentence-transformers/msmarco-distilbert-dot-v5"
print(f"Loading {MODEL2_NAME}...")
model2 = SentenceTransformer(MODEL2_NAME, device=device)
print("Model loaded.")

wikir_results_m2 = wikir_retrieval(model2, MODEL2_NAME, doc_ids, doc_texts, test_qids, test_qtexts, device)
metrics_wikir_m2 = evaluate(wikir_results_m2, test_qrels)
print("\nWikIR — msmarco-distilbert-dot-v5:")
print(metrics_wikir_m2)

Loading sentence-transformers/msmarco-distilbert-dot-v5...


'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: e94092ac-534a-422f-b3c3-0a890a63848b)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/msmarco-distilbert-dot-v5/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


Model loaded.
Loading cached embeddings from /Users/uni/homework13April/embeddings/wikir_docs_sentence-transformers_msmarco-distilbert-dot-v5.npy
Encoding 100 queries...


Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Running semantic search...

WikIR — msmarco-distilbert-dot-v5:
{'P@1': 0.75, 'P@10': 0.18, 'P@20': 0.1175, 'MAP@20': 0.1442, 'nDCG@20': 0.3753}


In [11]:
## Model 2: msmarco-distilbert-dot-v5 — MIRAGE
mirage_results_m2, _ = mirage_retrieval(model2, MODEL2_NAME, mirage_dataset, mirage_docs_by_query, mirage_oracle, device)
metrics_mirage_m2 = evaluate(mirage_results_m2, mirage_qrels)
print("\nMIRAGE — msmarco-distilbert-dot-v5:")
print(metrics_mirage_m2)

MIRAGE msmarco-distilbert-dot-v5: 100%|██████████| 7560/7560 [08:14<00:00, 15.29it/s]


MIRAGE — msmarco-distilbert-dot-v5:
{'P@1': 0.8619, 'P@10': 0.1, 'P@20': 0.05, 'MAP@20': 0.924, 'nDCG@20': 0.9435}


In [12]:
# Results Summary Table
results_table = pd.DataFrame({
    "Model": [
        "multi-qa-MiniLM-L6-cos-v1",
        "msmarco-distilbert-dot-v5",
        "multi-qa-MiniLM-L6-cos-v1",
        "msmarco-distilbert-dot-v5",
    ],
    "Dataset": ["WikIR", "WikIR", "MIRAGE", "MIRAGE"],
    "P@1":     [metrics_wikir_m1["P@1"],   metrics_wikir_m2["P@1"],   metrics_mirage_m1["P@1"],   metrics_mirage_m2["P@1"]],
    "P@10":    [metrics_wikir_m1["P@10"],  metrics_wikir_m2["P@10"],  metrics_mirage_m1["P@10"],  metrics_mirage_m2["P@10"]],
    "P@20":    [metrics_wikir_m1["P@20"],  metrics_wikir_m2["P@20"],  metrics_mirage_m1["P@20"],  metrics_mirage_m2["P@20"]],
    "MAP@20":  [metrics_wikir_m1["MAP@20"],metrics_wikir_m2["MAP@20"],metrics_mirage_m1["MAP@20"],metrics_mirage_m2["MAP@20"]],
    "nDCG@20": [metrics_wikir_m1["nDCG@20"],metrics_wikir_m2["nDCG@20"],metrics_mirage_m1["nDCG@20"],metrics_mirage_m2["nDCG@20"]],
})

print("=== Task 1 — Neural Ranking Results ===")
print(results_table.to_string(index=False))

=== Task 1 — Neural Ranking Results ===
                    Model Dataset    P@1  P@10   P@20  MAP@20  nDCG@20
multi-qa-MiniLM-L6-cos-v1   WikIR 0.6300 0.184 0.1215  0.1368   0.3547
msmarco-distilbert-dot-v5   WikIR 0.7500 0.180 0.1175  0.1442   0.3753
multi-qa-MiniLM-L6-cos-v1  MIRAGE 0.8774 0.100 0.0500  0.9331   0.9503
msmarco-distilbert-dot-v5  MIRAGE 0.8619 0.100 0.0500  0.9240   0.9435
